##  Core Porosity Upscaling to Log Support

### Objective

Core plug porosity measurements represent centimeter-scale support, while wireline logs respond over a much larger vertical resolution (~0.5 m).

To make core and log data comparable (e.g., for machine learning training), we upscale core porosity to log support.

---

###  Upscaling Method

We model the log vertical resolution using a Gaussian tool response function.

**Parameters used:**

- FWHM (Full Width Half Maximum): 0.55 m  
- Log depth sampling: 0.10 m  
- Plug length (if not provided): 0.025 m  
- Cutoff: ±3 sigma  

FWHM is converted to standard deviation (sigma) using:

sigma = FWHM / (2 * sqrt(2 * ln(2)))

Each core plug is treated as a depth interval [TOP, BOTTOM].

For every depth z on the log grid, the upscaled porosity is computed as a weighted average:

phi_log(z) = sum( w_i(z) * phi_i ) / sum( w_i(z) )

The weights are calculated as:

w_i(z) = CDF( (BOTTOM_i - z) / sigma )  
         - CDF( (TOP_i - z) / sigma )

where CDF is the standard normal cumulative distribution function.

This effectively integrates the Gaussian kernel over each plug interval.

---

###  Additional Processing

- Core plugs are separated into depth chunks to prevent smoothing across large missing-core gaps.
- Logs are aggregated to a 0.1 m depth grid.
- Only plugs within ±3 sigma contribute to each evaluation depth.
- Low kernel-mass points are flagged as unreliable (quality control).

---

###  Output

For each well, the following are generated:

- CORE_PHI_LOGSUP — Upscaled core porosity (log support)
- CORE_KERNEL_MASS — Total kernel weight (support indicator)
- N_INTERVAL_USED — Number of contributing plug intervals
- Log curves (GR, DT, RHOB, NPHI, etc.)
- DensityPorosity (computed from RHOB)

The final training table is saved as:

/out/{WELL_NAME}_upscaled_training_table.csv

This dataset is used for supervised machine learning training.

In [ ]:
# --- Cell 1: imports & parameters ---
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import norm
import matplotlib.pyplot as plt

# Paths
DATA_DIR = Path("../data")     # change if needed
OUT_DIR  = Path("../out")      # where to save merged per-well tables
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Logging tool vertical support (FWHM in meters)
# Use ~0.3–0.5 for density/neutron; 0.6–1.0 for sonic; tune as needed.
FWHM_M = 0.55

# Target log grid spacing (meters)
DZ_LOG = 0.10

# Plug interval length if top/bottom not provided (meters). Typical plug ~1 inch ≈ 0.0254 m
PLUG_LEN_M = 0.025

# Smoothing strictness: ignore depths with little nearby core support
CUTOFF_SIGMA = 3.0           # ignore plugs beyond ±3σ
MIN_MASS_FRACTION = 0.10     # require >=10% of reference kernel mass within each chunk

# Column names
DEPTH_COL = "DEPTH"
PORO_COL  = "POROSITY"


In [ ]:
# --- Cell 2: utility functions ---

def fwhm_to_sigma(fwhm_m: float) -> float:
    """Convert Full-Width at Half-Maximum (FWHM) to Gaussian sigma."""
    return fwhm_m / (2.0 * np.sqrt(2.0 * np.log(2.0)))

def ensure_plug_intervals(df: pd.DataFrame,
                          center_col=DEPTH_COL,
                          top_col=None,
                          bot_col=None,
                          plug_len_m=PLUG_LEN_M) -> pd.DataFrame:
    """
    Add TOP/BOTTOM columns for each plug. If explicit top/bottom exist, use them.
    Otherwise build from center depth ± plug_len/2.
    """
    out = df.copy()
    if top_col and bot_col and top_col in out.columns and bot_col in out.columns:
        out["TOP"] = out[top_col].astype(float)
        out["BOTTOM"] = out[bot_col].astype(float)
    else:
        c = out[center_col].astype(float)
        L = float(plug_len_m)
        out["TOP"] = c - 0.5 * L
        out["BOTTOM"] = c + 0.5 * L
    # sanitize
    t = out[["TOP","BOTTOM"]].min(axis=1)
    b = out[["TOP","BOTTOM"]].max(axis=1)
    out["TOP"], out["BOTTOM"] = t, b
    return out

def aggregate_logs_to_grid(log_df: pd.DataFrame,
                           depth_col=DEPTH_COL,
                           dz=DZ_LOG) -> pd.DataFrame:
    """
    Aggregate raw logs to a clean dz grid by mean (per numeric column).
    Assumes a single well per file; we'll add Well name outside this function.
    """
    df = log_df.copy()
    df["_bin"] = (np.floor(df[depth_col] / dz) * dz).round(3)
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    agg = {c: "mean" for c in num_cols if c not in [depth_col]}
    out = df.groupby("_bin", as_index=False).agg(agg)
    out = out.rename(columns={"_bin": depth_col})
    # re-order
    cols = [depth_col] + [c for c in out.columns if c != depth_col]
    return out[cols]


In [ ]:
# --- Cell 3: core → log-support upscaling (Gaussian interval weights) ---

def core_to_log_support(
    plug_df: pd.DataFrame,
    eval_depths_df: pd.DataFrame,
    well_col="Well",
    chunk_col="CHUNK_ID",
    top_col="TOP",
    bot_col="BOTTOM",
    phi_col=PORO_COL,
    eval_depth_col=DEPTH_COL,
    fwhm_m=FWHM_M,
    cutoff_sigma=CUTOFF_SIGMA,
    min_mass_fraction=MIN_MASS_FRACTION,
) -> pd.DataFrame:
    """
    Compute log-support porosity at eval depths by integrating a Gaussian tool response
    over each plug interval. Smoothing is performed *within each chunk* only (no smearing
    across >3 m gaps).
    
    Weight per plug interval for depth z:
      w_i(z) = CDF((BOTTOM_i - z)/σ) - CDF((TOP_i - z)/σ)
    Then:
      phi_hat(z) = sum[w_i(z) * phi_i] / sum[w_i(z)]
    We also return CORE_KERNEL_MASS = sum[w_i(z)] and N_INTERVAL_USED.
    """
    sigma = fwhm_to_sigma(fwhm_m)
    out = []

    def process_group(g_plug, g_eval):
        if g_eval.empty or g_plug.empty:
            return None
        z_eval = g_eval[eval_depth_col].to_numpy(dtype=float)
        t = g_plug[top_col].to_numpy(dtype=float)
        b = g_plug[bot_col].to_numpy(dtype=float)
        phi = g_plug[phi_col].to_numpy(dtype=float)

        # reference mass (for threshold): kernel mass at plug centers
        centers = 0.5 * (t + b)
        w_test = norm.cdf((b[None, :] - centers[:, None]) / sigma) - norm.cdf((t[None, :] - centers[:, None]) / sigma)
        mass_ref = np.nanmax(np.sum(w_test, axis=1))
        if not np.isfinite(mass_ref) or mass_ref <= 0:
            mass_ref = 1.0

        rows = []
        for z in z_eval:
            z_lo = z - cutoff_sigma * sigma
            z_hi = z + cutoff_sigma * sigma
            mask = (b >= z_lo) & (t <= z_hi)
            if not np.any(mask):
                rows.append((z, np.nan, 0.0, 0))
                continue
            tt = t[mask]; bb = b[mask]; pp = phi[mask]
            w = norm.cdf((bb - z)/sigma) - norm.cdf((tt - z)/sigma)
            mass = w.sum()
            if mass < (min_mass_fraction * mass_ref):
                rows.append((z, np.nan, mass, mask.sum()))
                continue
            phi_hat = float(np.dot(w, pp) / mass)
            rows.append((z, phi_hat, mass, mask.sum()))

        g_out = g_eval.copy()
        g_out["CORE_PHI_LOGSUP"]  = [r[1] for r in rows]
        g_out["CORE_KERNEL_MASS"] = [r[2] for r in rows]
        g_out["N_INTERVAL_USED"]  = [r[3] for r in rows]
        return g_out[[well_col, eval_depth_col, "CORE_PHI_LOGSUP", "CORE_KERNEL_MASS", "N_INTERVAL_USED"]]

    plugs = plug_df.dropna(subset=[phi_col]).copy()
    # Fallback if CHUNK_ID missing
    if chunk_col not in plugs.columns:
        plugs["_CHUNK_TMP_"] = 0
        chunk_col_use = "_CHUNK_TMP_"
    else:
        chunk_col_use = chunk_col

    for (w, c), g_plug in plugs.groupby([well_col, chunk_col_use]):
        g_plug = g_plug.sort_values(top_col)
        g_eval = eval_depths_df[eval_depths_df[well_col] == w].copy()
        # optional pad to restrict evaluation near this chunk
        if not g_plug.empty:
            zmin = g_plug[top_col].min() - 1.0
            zmax = g_plug[bot_col].max() + 1.0
            g_eval = g_eval[(g_eval[eval_depth_col] >= zmin) & (g_eval[eval_depth_col] <= zmax)]
        part = process_group(g_plug, g_eval)
        if part is not None:
            part[chunk_col] = c if chunk_col_use == chunk_col else 0
            out.append(part)

    if out:
        return pd.concat(out, ignore_index=True)
    return pd.DataFrame(columns=[well_col, eval_depth_col, "CORE_PHI_LOGSUP", "CORE_KERNEL_MASS", "N_INTERVAL_USED", chunk_col])


In [ ]:
# --- Cell 4: per-well loading & preprocessing ---

def load_and_prepare_single_well(core_file: Path, logs_file: Path) -> tuple[pd.DataFrame, pd.DataFrame, str]:
    """
    Load a well's core and logs, build CHUNK_ID (>3 m gaps), add TOP/BOTTOM,
    and aggregate logs to 0.1 m grid. Returns (plugs_wi, logs_0p1, well_name).
    """
    well_name = core_file.stem.replace("_shifted_chunks", "")
    core_df = pd.read_csv(core_file)
    logs_df = pd.read_csv(logs_file)

    # Build CHUNK_ID: gaps > 3 m
    core_df = core_df.sort_values(DEPTH_COL).reset_index(drop=True)
    core_df["DEPTH_DIFF"] = core_df[DEPTH_COL].diff().fillna(0.0)
    core_df["CHUNK_ID"] = (core_df["DEPTH_DIFF"] > 3.0).cumsum()
    core_df = core_df.drop(columns=["DEPTH_DIFF"])
    core_df["Well"] = well_name

    # Ensure intervals
    plugs_wi = ensure_plug_intervals(core_df, center_col=DEPTH_COL, plug_len_m=PLUG_LEN_M)

    # Aggregate logs to 0.1 m
    logs_0p1 = aggregate_logs_to_grid(logs_df, depth_col=DEPTH_COL, dz=DZ_LOG)
    logs_0p1["Well"] = well_name

    # Compute DensityPorosity if desired and RHOB is present
    if "RHOB" in logs_0p1.columns:
        # Matrix=2.65, Fluid=1.0 (example); adjust for your lithology/fluid if needed
        logs_0p1["DensityPorosity"] = (2.65 - logs_0p1["RHOB"]) * 100.0 / (2.65 - 1.0)

    # Reorder
    cols = ["Well", DEPTH_COL] + [c for c in logs_0p1.columns if c not in ["Well", DEPTH_COL]]
    logs_0p1 = logs_0p1[cols]

    return plugs_wi, logs_0p1, well_name


In [ ]:
# --- Cell 5: multi-well driver ---

def build_upscaled_training_tables(data_dir: Path = DATA_DIR) -> pd.DataFrame:
    """
    For each well (matched *_shifted_chunks.csv with WELL.csv), build an upscaled
    training table at 0.1 m: logs + CORE_PHI_LOGSUP + weights.
    Returns concatenated DataFrame of all wells.
    Also saves each well's table into OUT_DIR.
    """
    core_files = sorted(data_dir.glob("*_shifted_chunks.csv"))
    all_tables = []

    for core_file in core_files:
        well_name = core_file.stem.replace("_shifted_chunks", "")
        logs_file = data_dir / f"{well_name}.csv"
        if not logs_file.exists():
            print(f"⚠️ Skipping {well_name} — log file not found: {logs_file.name}")
            continue

        plugs_wi, logs_0p1, well_name = load_and_prepare_single_well(core_file, logs_file)

        # Evaluate target at the 0.1 m log grid (only within chunk extents ±1 m)
        target = core_to_log_support(
            plug_df=plugs_wi,
            eval_depths_df=logs_0p1[["Well", DEPTH_COL]].copy(),
            well_col="Well",
            chunk_col="CHUNK_ID",
            top_col="TOP", bot_col="BOTTOM",
            phi_col=PORO_COL,
            eval_depth_col=DEPTH_COL,
            fwhm_m=FWHM_M,
            cutoff_sigma=CUTOFF_SIGMA,
            min_mass_fraction=MIN_MASS_FRACTION
        )

        # Merge logs + target
        train_df = logs_0p1.merge(target, on=["Well", DEPTH_COL], how="left")

        # Save per-well table
        out_path = OUT_DIR / f"{well_name}_upscaled_training_table.csv"
        train_df.to_csv(out_path, index=False)
        print(f"✓ Saved {out_path.name} with {len(train_df)} rows")

        all_tables.append(train_df)

    if all_tables:
        return pd.concat(all_tables, ignore_index=True)
    return pd.DataFrame()


In [ ]:
# --- Cell 6: run multi-well build & preview ---

all_train = build_upscaled_training_tables(DATA_DIR)

print("Combined shape:", all_train.shape)
display_cols = [c for c in ["Well", DEPTH_COL, "GR", "DT", "RHOB", "NPHI",
                            "DensityPorosity", "CORE_PHI_LOGSUP",
                            "CORE_KERNEL_MASS", "N_INTERVAL_USED"] if c in all_train.columns]
all_train.head(20)[display_cols]
